# GV Tools 0.29.6 — task-oriented ingestion demonstration

This master notebook uses only the supported v0.9 public namespaces:
`gv_tools.core`, `gv_tools.io`, `gv_tools.config`, `gv_tools.correct`, and
`gv_tools.graph`. It can be launched from any directory. Set `GV_TOOLS_HOME`
if the project is not located at `~/Desktop/Work/GV Tools`.

In [ ]:
from pathlib import Path
import os
import sys

GV_HOME = Path(
    os.environ.get("GV_TOOLS_HOME", Path.home() / "Desktop" / "Work" / "GV Tools")
).expanduser().resolve()
PACKAGE = GV_HOME / "gv_tools"
SOURCE = PACKAGE / "src"
if SOURCE.is_dir() and str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
if not PACKAGE.is_dir():
    raise FileNotFoundError(
        f"GV Tools project not found at {PACKAGE}. "
        "Set GV_TOOLS_HOME to the directory containing gv_tools and Samples."
    )

import gv_tools

if gv_tools.__version__ != "0.29.6":
    raise RuntimeError(
        f"This notebook requires GV Tools 0.29.6, but the Jupyter kernel has "
        f"{gv_tools.__version__} loaded. Restart the kernel, then Run All cells."
    )

SAMPLES = Path(os.environ.get("GV_TOOLS_SAMPLES", GV_HOME / "Samples")).expanduser()
OUTPUT = Path(os.environ.get("GV_TOOLS_OUTPUT", GV_HOME / "Demo_Output")).expanduser()
OUTPUT.mkdir(parents=True, exist_ok=True)

print("GV Tools:", gv_tools.__version__)
print("Loaded from:", Path(gv_tools.__file__).resolve())
print("Project home:", GV_HOME)
print("Samples:", SAMPLES)
print("Output:", OUTPUT)
print("Adapters:", gv_tools.config.available_adapters())

## Dependency availability

The report checks required and optional dependency groups without installing
or modifying the environment.

In [ ]:
gv_tools.config.dependency_report(groups=("required", "parsivel", "radar", "netcdf", "plot"))

## PIERS RM Young All-in-One ingest

The canonical sample demonstrates raw decoding, normalized xarray output,
validation, and NetCDF/CSV writing.

In [ ]:
AIO_DAY = "2026-07-21"
AIO_FILES = [SAMPLES / "PIERS0027_WX_20260721000005.csv"]
print("Available dates:", gv_tools.io.discover_aio(AIO_FILES))
raw_aio = gv_tools.io.read_aio_raw(AIO_FILES)
print("Source files:", raw_aio.source_files)
print("Rejected rows:", raw_aio.rejected_rows)
print("Duplicate rows:", raw_aio.duplicate_rows)
raw_aio.observations.head()


In [ ]:
aio = gv_tools.io.read_aio(AIO_FILES)
aio_output = gv_tools.io.write_product(
    aio, OUTPUT, formats=("netcdf", "csv"), day=AIO_DAY, engine="h5netcdf"
)
print(aio_output)
aio


## All-in-One quicklooks

The 3×2 dashboard combines temperature, pressure, humidity, wind time series, a wind rose, and a data summary.

In [ ]:
quicklook_path = (
    gv_tools.graph.plot_directory(OUTPUT, "RM_Young_AIO_Quicklook", AIO_DAY)
    / f"WFF_PIERS0027_{AIO_DAY.replace('-', '')}_quicklook.png"
)
gv_tools.graph.plot_aio_quicklook(
    aio, savefig=quicklook_path, title=f"WFF PIERS0027 — {AIO_DAY}"
)


## Optional Met One All-in-One ingest

Set `GV_TOOLS_METONE_FILE` to a monthly Met One file. The cell skips cleanly
when the file is not available.

In [ ]:
metone_file = Path(
    os.environ.get("GV_TOOLS_METONE_FILE", "/Volumes/TBW/distro/aio/AIO_GAIL_202501.dat")
).expanduser()
if not metone_file.is_file():
    print(f"Skipped: Met One file not found: {metone_file}")
else:
    metone_day = "2025-01-15"
    metone = gv_tools.io.read_aio([metone_file], XARRAY=True)
    print(metone)


## Optional WS800 full-data quicklook

Set `GV_TOOLS_WS800_FILES` to comma-separated WS800 CSV paths. The quicklook accepts either the default DataFrame or an xarray Dataset and plots six stacked field families: air temperature, dew point, relative humidity, absolute humidity, mixing ratio, and lightning. Every GV Tools plot title includes site, observation date, and instrument. Line-plot legends use Matplotlib's automatic `best` placement.

In [ ]:
ws800_files = os.environ.get("GV_TOOLS_WS800_FILES")
if not ws800_files:
    print("Skipped: set GV_TOOLS_WS800_FILES to comma-separated file paths")
else:
    ws800_files = [Path(name).expanduser() for name in ws800_files.split(",") if name]
    ws800_full = gv_tools.io.read_ws800_full(ws800_files)
    gv_tools.graph.plot_ws800_full_quicklook(ws800_full)


## Optional Parsivel ingest

Set `GV_TOOLS_PARSIVEL_FILES` to comma-separated, fully qualified PIERS or APU filenames ordered from earliest to latest. GV Tools derives its internal processing interval from the first and last filenames. The optional `process-parsivel` dependency is required for raw Parsivel input.

In [ ]:
parsivel_files = os.environ.get("GV_TOOLS_PARSIVEL_FILES")
if not parsivel_files:
    print("Skipped: set GV_TOOLS_PARSIVEL_FILES to comma-separated file paths")
else:
    parsivel_files = [Path(name).expanduser() for name in parsivel_files.split(",") if name]
    parsivel = gv_tools.io.read_parsivel(parsivel_files)
    print(parsivel)
    parsivel_day = str(parsivel.time.values[0])[:10]
    parsivel_image = os.environ.get("GV_TOOLS_PARSIVEL_FIGURE")
    parsivel_figure = gv_tools.graph.plot_parsivel_quicklook(
        parsivel,
        day=parsivel_day,
        cmap="viridis",
        colorbar_location="right",
        savefig=Path(parsivel_image).expanduser() if parsivel_image else None,
    )


## Optional MRR2 and MRRPro ingest

The default paths demonstrate both MRR models when their sample archives are mounted.
Override either path with `GV_TOOLS_MRR2_FILE` or `GV_TOOLS_MRRPRO_FILE`.


In [ ]:
mrr_files = {
    "MRR2": Path(os.environ.get(
        "GV_TOOLS_MRR2_FILE",
        "/Volumes/TBW/distro/mrr/mrr2-02/NetCDF/202607/0722.ave.nc.zip",
    )).expanduser(),
    "MRRPro": Path(os.environ.get(
        "GV_TOOLS_MRRPRO_FILE",
        "/Volumes/TBW/distro/mrr/mrrpro-08/NetCDF/202607/20260722/20260722_070000.nc.zip",
    )).expanduser(),
}
for expected_model, mrr_path in mrr_files.items():
    if not mrr_path.is_file():
        print(f"Skipped: {expected_model} file not found: {mrr_path}")
        continue
    mrr = gv_tools.io.read_mrr(mrr_path)
    print(mrr)
    print("Model:", mrr.attrs["mrr_model"])
    print("Time coverage:", mrr.time.values[[0, -1]])
    rain_rate = mrr.MRR_RR if expected_model == "MRR2" else mrr.RR
    print("Rain rate:", rain_rate)
    profile_fields = (
        ["MRR_Capital_Z", "MRR_RR", "MRR_LWC", "MRR_W"]
        if expected_model == "MRR2" else ["Ze", "RR", "LWC", "VEL"]
    )
    mrr_figure_dir = os.environ.get("GV_TOOLS_MRR_FIGURE_DIR")
    mrr_savefig = (
        Path(mrr_figure_dir).expanduser() / f"{expected_model}_time_height.png"
        if mrr_figure_dir else None
    )
    gv_tools.graph.plot_mrr_time_height_quicklook(
        mrr, profile_fields, height_range_km=(0, 3), savefig=mrr_savefig
    )


## Optional radar ingest

Set `GV_TOOLS_RADAR_FILE` to an uncompressed SIGMET/IRIS or CF/Radial file.
Install the `radar` extra to provide ARM Py-ART and xradar. Py-ART is the default; pass `XRADAR=True` to load through xradar and receive a Py-ART-compatible radar object. `file_field_names` is Py-ART-specific; when supplied with `XRADAR=True`, GV Tools warns and ignores it.


In [ ]:
radar_name = os.environ.get("GV_TOOLS_RADAR_FILE")
if not radar_name:
    print("Skipped: set GV_TOOLS_RADAR_FILE to run this section")
else:
    radar_path = Path(radar_name).expanduser()
    route = gv_tools.io.inspect_radar(radar_path)
    radar = gv_tools.io.read_radar(
        radar_path, file_field_names=True, XRADAR=False,
    )
    import pyart
    radar_output = OUTPUT / f"{radar_path.name.removesuffix('.gz')}.gv_tools.cf"
    pyart.io.write_cfradial(radar_output, radar, format='NETCDF4')
    print('CF/Radial output:', radar_output)
    print(route)
    print(f"{radar.nsweeps} sweeps, {radar.nrays} rays, {radar.ngates} gates")
    print("Fields:", sorted(radar.fields))
    radar_sweep = int(os.environ.get("GV_TOOLS_RADAR_SWEEP", "0"))
    radar_field = os.environ.get("GV_TOOLS_RADAR_FIELD")
    radar_image = os.environ.get("GV_TOOLS_RADAR_FIGURE")
    radar_savefig = Path(radar_image).expanduser() if radar_image else None
    scan_type = radar.metadata.get("gv_tools_scan_type")
    if scan_type == "PPI":
        gv_tools.graph.plot_radar_ppi_quicklook(
            radar, field=radar_field, sweep=radar_sweep,
            radial_spoke_interval_deg=30, range_ring_interval_km=25,
            savefig=radar_savefig,
        )
    elif scan_type == "RHI":
        gv_tools.graph.plot_radar_rhi_quicklook(
            radar, field=radar_field, sweep=radar_sweep,
            savefig=radar_savefig,
        )
    elif scan_type == "BB":
        gv_tools.graph.plot_radar_bb_zdr_calibration(
            radar, field=radar_field, sweep=radar_sweep,
            savefig=radar_savefig,
        )
    else:
        print(f"No quicklook is defined for radar scan type {scan_type}")


## Demonstrated v0.22 API

- `gv_tools.core`: instrument metadata
- `gv_tools.io`: adapters, MRR2/MRRPro and radar ingest, and product writing
- `gv_tools.config`: adapters and dependency reporting
- `gv_tools.correct`: product validation
- `gv_tools.graph`: quicklooks and plot directories

## 2DVD (GV Tools 0.30.0)
Set `dvd_file` to a plain VYYDDD.drops.txt file or a ZIP/TGZ containing one day.
Ingest preserves raw records; calculations return both velocity choices.
Export and figure saving are explicit. See the 2DVD manual for QC and units.

In [ ]:
from pathlib import Path
from gv_tools.io import read_2dvd, save_products
from gv_tools.core import calculate_products
from gv_tools.graph import plot_integral_parameters, plot_dsd

dvd_file = None  # Replace with Path("/path/to/V23022.drops.txt")
if dvd_file is not None:
    raw_dvd = read_2dvd(dvd_file, site="WFF", instrument="sn37")
    dvd_parameters, dvd_dsd = calculate_products(raw_dvd)
    rain_figure = plot_integral_parameters(dvd_parameters, velocity="measured")
    dsd_figure = plot_dsd(dvd_dsd, velocity="terminal")
    # paths = save_products(dvd_parameters, dvd_dsd, Path.home() / "Desktop/Work/GV Tools/Output")
else:
    print("Set dvd_file to process a 2DVD day.")
